In [14]:
"""
FILE 4: ENSEMBLE MODELS
Run this on Colab Account 4
Trains: Voting Ensembles, Stacking Ensembles, Weighted Ensembles
NOTE: This requires the best models from Files 1-3 to be loaded
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import gc  # Garbage collection to free memory

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                            confusion_matrix, balanced_accuracy_score, classification_report)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (ExtraTreesClassifier, RandomForestClassifier,
                              VotingClassifier, StackingClassifier, GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from scipy.sparse import hstack, csr_matrix
from sklearn.utils.class_weight import compute_class_weight
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*100)
print("🗳️ FILE 4: ENSEMBLE MODELS (FIXED - MEMORY EFFICIENT)")
print("="*100)

🗳️ FILE 4: ENSEMBLE MODELS (FIXED - MEMORY EFFICIENT)


In [15]:
# ==================== DATA LOADING & PREPROCESSING ====================
print("\n📂 Loading data...")
df = pd.read_excel("bharatfakenewskosh.xlsx")

statement_col = 'Eng_Trans_Statement'
body_col = 'Eng_Trans_News_Body'
label_col = 'Label'

df = df[[statement_col, body_col, label_col]].dropna()

def clean_text_advanced(text):
    text = str(text).lower()
    if ':' in text:
        parts = text.split(':', 1)
        if any(word in parts[0] for word in ['fact-check', 'fact check', 'wrong', 'false']):
            text = parts[1]

    leak_words = ['fact-check', 'factcheck', 'debunked', 'hoax', 'busted', 'fake', 'false claim']
    for phrase in leak_words:
        text = text.replace(phrase, ' ')

    text = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', text)
    text = re.sub(r'!{2,}', ' MULTIEXCLAIM ', text)
    text = re.sub(r'\?{2,}', ' MULTIQUESTION ', text)
    text = re.sub(r'\.{3,}', ' ELLIPSIS ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['statement_clean'] = df[statement_col].apply(clean_text_advanced)
df['body_clean'] = df[body_col].apply(clean_text_advanced)
df['combined_text'] = df['statement_clean'] + ' [SEP] ' + df['body_clean']

df['Label'] = df[label_col].map({'TRUE': 1, 'FALSE': 0}) if df[label_col].dtype == 'object' else df[label_col].astype(int)
df = df.dropna(subset=['Label'])
df['combined_words'] = df['combined_text'].str.split().str.len()
df = df[df['combined_words'] >= 15]
df = df.drop_duplicates(subset=['combined_text'], keep='first')

print(f"✅ Dataset: {len(df)} samples")

# ==================== FEATURE ENGINEERING ====================
print("\n🔧 Creating enhanced feature sets...")

def extract_advanced_features(df):
    features = pd.DataFrame()
    features['stmt_len'] = df['statement_clean'].str.len()
    features['body_len'] = df['body_clean'].str.len()
    features['combined_len'] = df['combined_text'].str.len()
    features['stmt_words'] = df['statement_clean'].str.split().str.len()
    features['body_words'] = df['body_clean'].str.split().str.len()
    features['total_words'] = features['stmt_words'] + features['body_words']
    features['stmt_body_ratio'] = features['stmt_len'] / (features['body_len'] + 1)
    features['word_density'] = features['total_words'] / (features['combined_len'] + 1)
    features['avg_word_len'] = features['combined_len'] / (features['total_words'] + 1)
    features['exclaim_count'] = df['combined_text'].str.count('!')
    features['question_count'] = df['combined_text'].str.count('\?')
    features['period_count'] = df['combined_text'].str.count('\.')
    features['comma_count'] = df['combined_text'].str.count(',')
    features['quote_count'] = df['combined_text'].str.count('"') + df['combined_text'].str.count("'")
    features['caps_count'] = df['combined_text'].str.count('[A-Z]')
    features['digit_count'] = df['combined_text'].str.count('\d')
    features['has_multiexclaim'] = df['combined_text'].str.contains('MULTIEXCLAIM').astype(int)
    features['has_multiquestion'] = df['combined_text'].str.contains('MULTIQUESTION').astype(int)
    features['has_ellipsis'] = df['combined_text'].str.contains('ELLIPSIS').astype(int)
    features['caps_ratio'] = features['caps_count'] / (features['combined_len'] + 1)
    features['digit_ratio'] = features['digit_count'] / (features['combined_len'] + 1)

    fake_keywords = ['viral', 'shocking', 'breaking', 'exposed', 'revealed',
                     'truth', 'must watch', 'exclusive', 'urgent', 'alert',
                     'proof', 'evidence', 'conspiracy', 'hidden', 'secret']
    for keyword in fake_keywords:
        features[f'has_{keyword}'] = df['combined_text'].str.contains(keyword, regex=False).astype(int)

    features['sentence_count'] = df['combined_text'].str.count(r'[.!?]') + 1
    features['avg_sentence_len'] = features['total_words'] / features['sentence_count']

    return features

linguistic_features = extract_advanced_features(df)
print(f"✅ Linguistic features: {linguistic_features.shape[1]} features")

X = df['combined_text']
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
ling_train, ling_test = train_test_split(linguistic_features, test_size=0.2, stratify=y, random_state=42)

print("\n📄 Creating text vectorizers...")
vectorizer_tfidf_word = TfidfVectorizer(max_features=20000, ngram_range=(1,3),
                                        min_df=2, max_df=0.9, sublinear_tf=True)
X_train_tfidf_word = vectorizer_tfidf_word.fit_transform(X_train)
X_test_tfidf_word = vectorizer_tfidf_word.transform(X_test)

vectorizer_tfidf_char = TfidfVectorizer(max_features=8000, ngram_range=(2,6),
                                        analyzer='char', min_df=2, max_df=0.9)
X_train_tfidf_char = vectorizer_tfidf_char.fit_transform(X_train)
X_test_tfidf_char = vectorizer_tfidf_char.transform(X_test)

vectorizer_count = CountVectorizer(max_features=15000, ngram_range=(1,2),
                                   min_df=2, max_df=0.9, binary=True)
X_train_count = vectorizer_count.fit_transform(X_train)
X_test_count = vectorizer_count.transform(X_test)

scaler = StandardScaler()
ling_train_scaled = scaler.fit_transform(ling_train)
ling_test_scaled = scaler.transform(ling_test)

X_train_combined = hstack([X_train_tfidf_word, X_train_tfidf_char,
                           X_train_count, csr_matrix(ling_train_scaled)])
X_test_combined = hstack([X_test_tfidf_word, X_test_tfidf_char,
                          X_test_count, csr_matrix(ling_test_scaled)])

print(f"✅ Combined features: {X_train_combined.shape[1]} features")

print("\n📊 Applying SMOTE for class balancing...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_combined, y_train)
print(f"✅ After SMOTE: {X_train_resampled.shape[0]} samples")

class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
scale_pos_weight = class_weights[1]/class_weights[0]


📂 Loading data...
✅ Dataset: 24455 samples

🔧 Creating enhanced feature sets...
✅ Linguistic features: 38 features

📄 Creating text vectorizers...
✅ Combined features: 43038 features

📊 Applying SMOTE for class balancing...
✅ After SMOTE: 23794 samples


In [16]:
# ==================== TRAIN SIMPLE MODELS FIRST ====================
print("\n" + "="*100)
print("📊 TRAINING: SIMPLE BASELINE MODELS")
print("="*100)

results = {}
trained_models = {}

# 1. Simple Logistic Regression
print("\n1️⃣ Training: Logistic Regression (Simple)")
start_time = time.time()
lr_simple = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='saga',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
lr_simple.fit(X_train_resampled, y_train_resampled)
train_time = time.time() - start_time
preds = lr_simple.predict(X_test_combined)

results["Logistic Regression (Simple)"] = {
    'Accuracy': accuracy_score(y_test, preds),
    'Balanced_Acc': balanced_accuracy_score(y_test, preds),
    'Precision': precision_score(y_test, preds),
    'Recall': recall_score(y_test, preds),
    'F1': f1_score(y_test, preds),
    'Time': train_time
}
trained_models["Logistic Regression (Simple)"] = lr_simple
print(f"   ✅ Acc: {results['Logistic Regression (Simple)']['Accuracy']:.4f} | Time: {train_time:.1f}s")

# 2. Linear SVM
print("\n2️⃣ Training: Linear SVM")
start_time = time.time()
svm_linear = LinearSVC(
    C=1.0,
    max_iter=2000,
    dual=False,
    class_weight='balanced',
    random_state=42
)
svm_linear.fit(X_train_resampled, y_train_resampled)
train_time = time.time() - start_time
preds = svm_linear.predict(X_test_combined)

results["Linear SVM"] = {
    'Accuracy': accuracy_score(y_test, preds),
    'Balanced_Acc': balanced_accuracy_score(y_test, preds),
    'Precision': precision_score(y_test, preds),
    'Recall': recall_score(y_test, preds),
    'F1': f1_score(y_test, preds),
    'Time': train_time
}
trained_models["Linear SVM"] = svm_linear
print(f"   ✅ Acc: {results['Linear SVM']['Accuracy']:.4f} | Time: {train_time:.1f}s")

# Free memory
gc.collect()


📊 TRAINING: SIMPLE BASELINE MODELS

1️⃣ Training: Logistic Regression (Simple)
   ✅ Acc: 0.5383 | Time: 153.1s

2️⃣ Training: Linear SVM
   ✅ Acc: 0.5377 | Time: 73.9s


3976

In [17]:
print("\n🔧 Creating memory-efficient base models for ensembles...")

# Use smaller Extra Trees for ensemble (to save memory)
base_et = ExtraTreesClassifier(
    n_estimators=300,  # Reduced from 700
    max_depth=40,      # Reduced from 60
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=False,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
    verbose=0
)

base_rf = RandomForestClassifier(
    n_estimators=300,  # Reduced from 1000
    max_depth=40,      # Reduced from 70
    min_samples_split=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

# MEMORY FIX: Use hist tree method for XGBoost (much more memory efficient)
base_xgb = XGBClassifier(
    n_estimators=200,  # Reduced from 500
    max_depth=6,       # Reduced from 8
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',  # KEY FIX: More memory efficient
    n_jobs=1,            # KEY FIX: Single thread to avoid memory multiplication
    random_state=42,
    eval_metric='logloss'
)

base_lgbm = LGBMClassifier(
    n_estimators=200,  # Reduced from 500
    max_depth=6,       # Reduced from 8
    learning_rate=0.1,
    class_weight='balanced',
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=1,          # Single thread
    random_state=42,
    verbose=-1
)

base_cat = CatBoostClassifier(
    iterations=200,    # Reduced from 500
    depth=6,           # Reduced from 8
    learning_rate=0.1,
    auto_class_weights='Balanced',
    verbose=0,
    random_state=42,
    thread_count=1     # Single thread
)

print("✅ Base models created (memory-efficient versions)")



🔧 Creating memory-efficient base models for ensembles...
✅ Base models created (memory-efficient versions)


In [18]:
print("\n" + "="*100)
print("🗳️ TRAINING: ENSEMBLE MODELS (SEQUENTIAL - MEMORY SAFE)")
print("="*100)

# KEY FIX: Train models sequentially instead of parallel to avoid memory issues
print("\n⚠️  Training ensembles sequentially to avoid memory errors...")
print("This will take longer but won't crash.\n")

# Voting Ensemble 1: Soft voting (without XGBoost to save memory)
print("1️⃣ Training: Voting Ensemble (Trees + LGBM + Cat - Soft)")
try:
    # Train base models one by one
    print("   Training Extra Trees...")
    et_for_voting = ExtraTreesClassifier(**base_et.get_params())
    et_for_voting.fit(X_train_resampled, y_train_resampled)
    
    print("   Training Random Forest...")
    rf_for_voting = RandomForestClassifier(**base_rf.get_params())
    rf_for_voting.fit(X_train_resampled, y_train_resampled)
    
    print("   Training LightGBM...")
    lgbm_for_voting = LGBMClassifier(**base_lgbm.get_params())
    lgbm_for_voting.fit(X_train_resampled, y_train_resampled)
    
    print("   Training CatBoost...")
    cat_for_voting = CatBoostClassifier(**base_cat.get_params())
    cat_for_voting.fit(X_train_resampled, y_train_resampled)
    
    # Create voting ensemble with pre-trained models
    voting_soft = VotingClassifier(
        estimators=[
            ('et', et_for_voting),
            ('rf', rf_for_voting),
            ('lgbm', lgbm_for_voting),
            ('cat', cat_for_voting)
        ],
        voting='soft',
        n_jobs=1  # Single thread
    )
    
    # Fit only sets up the ensemble (models already trained)
    voting_soft.estimators_ = [et_for_voting, rf_for_voting, lgbm_for_voting, cat_for_voting]
    voting_soft.named_estimators_ = {
        'et': et_for_voting,
        'rf': rf_for_voting,
        'lgbm': lgbm_for_voting,
        'cat': cat_for_voting
    }
    voting_soft.le_ = LogisticRegression().fit([[0], [1]], [0, 1])  # Dummy label encoder
    voting_soft.classes_ = np.array([0, 1])
    
    preds = voting_soft.predict(X_test_combined)
    
    results["Voting: Trees+LGBM+Cat (Soft)"] = {
        'Accuracy': accuracy_score(y_test, preds),
        'Balanced_Acc': balanced_accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1': f1_score(y_test, preds),
        'Time': 0
    }
    trained_models["Voting: Trees+LGBM+Cat (Soft)"] = voting_soft
    print(f"   ✅ Acc: {results['Voting: Trees+LGBM+Cat (Soft)']['Accuracy']:.4f}")
    
    gc.collect()  # Free memory
except Exception as e:
    print(f"   ❌ Failed: {str(e)[:100]}")

# Voting Ensemble 2: Hard voting
print("\n2️⃣ Training: Voting Ensemble (Trees + LGBM - Hard)")
try:
    voting_hard = VotingClassifier(
        estimators=[
            ('et', et_for_voting),
            ('rf', rf_for_voting),
            ('lgbm', lgbm_for_voting)
        ],
        voting='hard',
        n_jobs=1
    )
    
    voting_hard.estimators_ = [et_for_voting, rf_for_voting, lgbm_for_voting]
    voting_hard.named_estimators_ = {
        'et': et_for_voting,
        'rf': rf_for_voting,
        'lgbm': lgbm_for_voting
    }
    voting_hard.le_ = LogisticRegression().fit([[0], [1]], [0, 1])
    voting_hard.classes_ = np.array([0, 1])
    
    preds = voting_hard.predict(X_test_combined)
    
    results["Voting: Trees+LGBM (Hard)"] = {
        'Accuracy': accuracy_score(y_test, preds),
        'Balanced_Acc': balanced_accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1': f1_score(y_test, preds),
        'Time': 0
    }
    trained_models["Voting: Trees+LGBM (Hard)"] = voting_hard
    print(f"   ✅ Acc: {results['Voting: Trees+LGBM (Hard)']['Accuracy']:.4f}")
    
    gc.collect()
except Exception as e:
    print(f"   ❌ Failed: {str(e)[:100]}")

# Stacking 1: LR meta-learner (lightweight)
print("\n3️⃣ Training: Stacking Ensemble (LR Meta-learner)")
try:
    stack_lr = StackingClassifier(
        estimators=[
            ('et', base_et),
            ('lgbm', base_lgbm)
        ],
        final_estimator=LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42),
        cv=3,  # Reduced from 5
        n_jobs=1  # Single thread
    )
    start_time = time.time()
    stack_lr.fit(X_train_resampled, y_train_resampled)
    train_time = time.time() - start_time
    preds = stack_lr.predict(X_test_combined)
    
    results["Stacking: ET+LGBM + LR"] = {
        'Accuracy': accuracy_score(y_test, preds),
        'Balanced_Acc': balanced_accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1': f1_score(y_test, preds),
        'Time': train_time
    }
    trained_models["Stacking: ET+LGBM + LR"] = stack_lr
    print(f"   ✅ Acc: {results['Stacking: ET+LGBM + LR']['Accuracy']:.4f} | Time: {train_time:.1f}s")
    
    gc.collect()
except Exception as e:
    print(f"   ❌ Failed: {str(e)[:100]}")

# Weighted Ensemble (lightweight - just probabilities)
print("\n4️⃣ Creating: Weighted Ensemble (Pre-trained models)")
try:
    # Get probabilities from already trained models
    et_proba = et_for_voting.predict_proba(X_test_combined)[:, 1]
    rf_proba = rf_for_voting.predict_proba(X_test_combined)[:, 1]
    lgbm_proba = lgbm_for_voting.predict_proba(X_test_combined)[:, 1]
    cat_proba = cat_for_voting.predict_proba(X_test_combined)[:, 1]
    
    # Weight by performance (equal for now, but can optimize)
    weights = np.array([0.25, 0.25, 0.25, 0.25])
    weighted_proba = (weights[0] * et_proba + 
                      weights[1] * rf_proba + 
                      weights[2] * lgbm_proba + 
                      weights[3] * cat_proba)
    weighted_preds = (weighted_proba > 0.5).astype(int)
    
    results["Weighted: All Trees Average"] = {
        'Accuracy': accuracy_score(y_test, weighted_preds),
        'Balanced_Acc': balanced_accuracy_score(y_test, weighted_preds),
        'Precision': precision_score(y_test, weighted_preds),
        'Recall': recall_score(y_test, weighted_preds),
        'F1': f1_score(y_test, weighted_preds),
        'Time': 0
    }
    print(f"   ✅ Acc: {results['Weighted: All Trees Average']['Accuracy']:.4f}")
except Exception as e:
    print(f"   ❌ Failed: {str(e)[:100]}")



🗳️ TRAINING: ENSEMBLE MODELS (SEQUENTIAL - MEMORY SAFE)

⚠️  Training ensembles sequentially to avoid memory errors...
This will take longer but won't crash.

1️⃣ Training: Voting Ensemble (Trees + LGBM + Cat - Soft)
   Training Extra Trees...
   Training Random Forest...
   Training LightGBM...
   Training CatBoost...
   ❌ Failed: 'LogisticRegression' object has no attribute 'inverse_transform'

2️⃣ Training: Voting Ensemble (Trees + LGBM - Hard)
   ❌ Failed: 'LogisticRegression' object has no attribute 'inverse_transform'

3️⃣ Training: Stacking Ensemble (LR Meta-learner)
   ✅ Acc: 0.6079 | Time: 338.0s

4️⃣ Creating: Weighted Ensemble (Pre-trained models)
   ✅ Acc: 0.6220


In [19]:
# ==================== RESULTS ANALYSIS ====================
print("\n" + "="*100)
print("📊 FINAL RESULTS - ALL MODELS")
print("="*100)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('F1', ascending=False)

print("\n🏆 TOP 10 MODELS (Ranked by F1):")
print("="*100)
print(results_df.head(10).to_string())

# Save results
results_df.to_csv('results_part4_ensembles_fixed.csv')
print("\n✅ Saved: results_part4_ensembles_fixed.csv")

best_model_name = results_df.index[0]
best_metrics = results_df.iloc[0]

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Accuracy:     {best_metrics['Accuracy']:.4f} ({best_metrics['Accuracy']*100:.2f}%)")
print(f"   Balanced Acc: {best_metrics['Balanced_Acc']:.4f}")
print(f"   Precision:    {best_metrics['Precision']:.4f}")
print(f"   Recall:       {best_metrics['Recall']:.4f}")
print(f"   F1 Score:     {best_metrics['F1']:.4f}")

# Save best model
if best_model_name in trained_models:
    joblib.dump(trained_models[best_model_name], 'best_model_part4_fixed.pkl')
    print(f"\n✅ Saved best model: best_model_part4_fixed.pkl")

# Detailed report for best model
print("\n" + "="*100)
print(f"📋 DETAILED CLASSIFICATION REPORT - {best_model_name}")
print("="*100)

if best_model_name in trained_models:
    best_model_obj = trained_models[best_model_name]
    best_preds = best_model_obj.predict(X_test_combined)
    print(classification_report(y_test, best_preds,
                              target_names=['Fake News (0)', 'True News (1)'],
                              digits=4))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, best_preds)
    tn, fp, fn, tp = cm.ravel()
    
    print("\n📊 Confusion Matrix:")
    print(f"   True Negatives:  {tn:4d}")
    print(f"   False Positives: {fp:4d}")
    print(f"   False Negatives: {fn:4d}")
    print(f"   True Positives:  {tp:4d}")

print("\n" + "="*100)
print("✅ FILE 4 COMPLETE (MEMORY-EFFICIENT VERSION)!")
print("="*100)
print(f"""
KEY FIXES APPLIED:
✓ Added Logistic Regression and Linear SVM
✓ XGBoost tree_method='hist' (more memory efficient)
✓ Reduced n_jobs to 1 (avoid memory multiplication)
✓ Sequential training instead of parallel
✓ Smaller ensemble models (300 trees instead of 700)
✓ Manual garbage collection after each model
✓ Removed most memory-intensive combinations

RESULT: Should run without memory errors!
""")


📊 FINAL RESULTS - ALL MODELS

🏆 TOP 10 MODELS (Ranked by F1):
                              Accuracy  Balanced_Acc  Precision    Recall        F1        Time
Weighted: All Trees Average   0.621959      0.531363   0.624200  0.950572  0.753565    0.000000
Stacking: ET+LGBM + LR        0.607851      0.546182   0.635733  0.831540  0.720571  337.999488
Linear SVM                    0.537722      0.522444   0.626643  0.593141  0.609432   73.851957
Logistic Regression (Simple)  0.538336      0.523875   0.627949  0.590787  0.608801  153.126626

✅ Saved: results_part4_ensembles_fixed.csv

🏆 BEST MODEL: Weighted: All Trees Average
   Accuracy:     0.6220 (62.20%)
   Balanced Acc: 0.5314
   Precision:    0.6242
   Recall:       0.9506
   F1 Score:     0.7536

📋 DETAILED CLASSIFICATION REPORT - Weighted: All Trees Average

✅ FILE 4 COMPLETE (MEMORY-EFFICIENT VERSION)!

KEY FIXES APPLIED:
✓ Added Logistic Regression and Linear SVM
✓ XGBoost tree_method='hist' (more memory efficient)
✓ Reduced n_jo